In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
from typing import Tuple

# ============================================
# STYLING CONSTANTS
# ============================================
PINK = "#C11C84"
NODE_BLACK = "#141D2B"
HACKER_GREY = "#A4B1CD"
WHITE = "#FFFFFF"
HTB_GREEN = "#2E8B57"
MALWARE_RED = "#FF6B6B"
AZURE = "#45B7D1"
NUGGET_YELLOW = "#F7DC6F"

# ============================================
# CLASS NAMES
# ============================================
CLASS_NAMES = ['adware', 'backdoor', 'benign', 'downloader',
               'spyware', 'trojan', 'virus', 'worm']

COLORS = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A',
          '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E2']

# ============================================
# CONFIGURATION
# ============================================
DATA_PATH = "/home/beylessen/Desktop/PFA2/Sorted"
MODEL_PATH = "malware_classifier.pth"
OUTPUT_DIR = "deepfool_output"
BATCH_SIZE = 1
NUM_EXAMPLES = 30  # Number of samples to attack

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "adversarial_images"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "original_images"), exist_ok=True)



In [2]:
# ============================================
# MODEL ARCHITECTURE
# ============================================
class MalwareClassifier(nn.Module):
    def __init__(self, n_classes):
        super(MalwareClassifier, self).__init__()
        self.resnet = models.resnet50(weights='DEFAULT')
        
        for param in self.resnet.parameters():
            param.requires_grad = False
        
        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Linear(num_features, 1000),
            nn.ReLU(),
            nn.Linear(1000, n_classes)
        )

    def forward(self, x):
        return self.resnet(x)



In [3]:
# ============================================
# DEEPFOOL ATTACK IMPLEMENTATION
# ============================================
def deepfool(image: torch.Tensor,
             net: nn.Module,
             num_classes: int = 8,
             overshoot: float = 0.02,
             max_iter: int = 50,
             device: str = 'cuda') -> Tuple[torch.Tensor, int, int, int, torch.Tensor]:
    """
    Generate minimal adversarial perturbation using DeepFool algorithm.

    Args:
        image (torch.Tensor): Input image tensor of shape (1, C, H, W)
        net (nn.Module): Target neural network in evaluation mode
        num_classes (int): Number of top-scoring classes to consider
        overshoot (float): Overshoot parameter for boundary crossing
        max_iter (int): Maximum iterations before terminating
        device (str): Computation device ('cuda' or 'cpu')

    Returns:
        Tuple containing:
            - r_tot (torch.Tensor): Total accumulated perturbation
            - loop_i (int): Number of iterations performed
            - label (int): Original predicted class
            - k_i (int): Final adversarial class
            - pert_image (torch.Tensor): Final perturbed image
    """
    image = image.to(device)
    net = net.to(device)
    
    # Original prediction and class ordering (descending score)
    f_image = net(image).data.cpu().numpy().flatten()
    I = f_image.argsort()[::-1]
    label = I[0]
    
    # Working tensors and accumulators
    input_shape = image.shape
    pert_image = image.clone()
    r_tot = torch.zeros(input_shape).to(device)
    loop_i = 0
    
    # Iterate until a successful perturbation is found or the limit is reached
    while loop_i < max_iter:
        x = pert_image.clone().requires_grad_(True)
        fs = net(x)
        
        # Current top prediction at x
        k_i = fs.data.cpu().numpy().flatten().argsort()[::-1][0]
        
        # Stop when the prediction changes
        if k_i != label:
            break
        
        # Initialize the best candidate step for this iteration
        pert = float('inf')
        w = None
        
        # Search minimal step among candidate classes
        for k in range(1, num_classes):
            if I[k] == label:
                continue
            
            # Compute gradient for candidate class
            if x.grad is not None:
                x.grad.zero_()
            fs[0, I[k]].backward(retain_graph=True)
            grad_k = x.grad.data.clone()
            
            # Compute gradient for original class
            if x.grad is not None:
                x.grad.zero_()
            fs[0, label].backward(retain_graph=True)
            grad_label = x.grad.data.clone()
            
            # Direction and distance under linearization
            w_k = grad_k - grad_label
            f_k = (fs[0, I[k]] - fs[0, label]).data.cpu().numpy()
            pert_k = abs(f_k) / (torch.norm(w_k.flatten()) + 1e-10)
            
            if pert_k < pert:
                pert = pert_k
                w = w_k
        
        # Minimal step for the selected direction
        r_i = (pert + 1e-4) * w / (torch.norm(w.flatten()) + 1e-10)
        r_tot = r_tot + r_i
        
        # Apply with overshoot to ensure crossing
        pert_image = image + (1 + overshoot) * r_tot
        loop_i += 1
    
    return r_tot, loop_i, label, k_i, pert_image



In [4]:
# ============================================
# LOAD MODEL AND DATA
# ============================================
def load_model_and_data(model_path, data_path, device):
    """Load trained model and test dataset"""
    print("[i] Loading model...")
    
    # Load the scripted model
    scripted_model = torch.jit.load(model_path, map_location=device)
    
    # Create new model and load weights
    model = MalwareClassifier(n_classes=8)
    model.load_state_dict(scripted_model.state_dict())
    model.to(device)
    model.eval()
    
    print("[✓] Model loaded successfully")
    
    # Load test dataset
    print("[i] Loading test dataset...")
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    test_dataset = datasets.ImageFolder(
        os.path.join(data_path, "test"),
        transform=transform
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4
    )
    
    print(f"[✓] Test dataset loaded: {len(test_dataset)} samples")
    
    return model, test_loader



In [5]:
# ============================================
# DENORMALIZATION FOR VISUALIZATION
# ============================================
def denormalize(tensor):
    """Denormalize tensor for visualization"""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean

# ============================================
# SAVE IMAGE
# ============================================
def save_image_to_disk(tensor, path):
    """Save tensor as image file"""
    img = denormalize(tensor.cpu()).squeeze()
    img = torch.clamp(img, 0, 1)
    img = img.permute(1, 2, 0).numpy()
    img = (img * 255).astype(np.uint8)
    Image.fromarray(img).save(path)



In [6]:
# ============================================
# BATCH ATTACK GENERATION
# ============================================
def generate_batch_attacks(model, test_loader, device, num_examples, target_class=None):
    """
    Generate adversarial examples for multiple samples
    
    Args:
        model: Trained malware classifier
        test_loader: DataLoader with test samples
        device: Computing device (cuda/cpu)
        num_examples: Number of samples to attack
        target_class: If specified (0-7), only attack samples from this class.
                     If None, attack samples from all classes.
    """
    if target_class is not None:
        print(f"\n{'='*60}")
        print(f"GENERATING {num_examples} ADVERSARIAL EXAMPLES")
        print(f"TARGET CLASS: {CLASS_NAMES[target_class].upper()}")
        print(f"{'='*60}\n")
    else:
        print(f"\n{'='*60}")
        print(f"GENERATING {num_examples} ADVERSARIAL EXAMPLES")
        print(f"TARGET CLASS: ALL CLASSES")
        print(f"{'='*60}\n")
    
    results = []
    success_count = 0
    processed_count = 0
    
    for idx, (data, target) in enumerate(test_loader):
        # Filter by target class if specified
        if target_class is not None:
            # Skip if this sample is not from the target class
            if target.item() != target_class:
                continue
        
        # Stop when we have enough samples
        if processed_count >= num_examples:
            break
        
        data = data.to(device)
        
        # Original prediction
        with torch.no_grad():
            original_output = model(data)
            original_pred = original_output.argmax(dim=1).item()
            original_confidence = F.softmax(original_output, dim=1).max().item()
        
        # Execute DeepFool attack
        r, iterations, orig_label, adv_label, pert_image = deepfool(
            data, model, num_classes=8, overshoot=0.02, max_iter=50, device=device
        )
        
        # Adversarial prediction
        with torch.no_grad():
            adv_output = model(pert_image)
            adv_confidence = F.softmax(adv_output, dim=1).max().item()
        
        # Track success
        success = (orig_label != adv_label)
        if success:
            success_count += 1
        
        # Compute metrics
        l2_norm = torch.norm(r.cpu()).item()
        linf_norm = torch.abs(r.cpu()).max().item()
        relative_pert = l2_norm / torch.norm(data.cpu()).item()
        
        # Save images to disk
        orig_img_path = os.path.join(OUTPUT_DIR, "original_images", f"sample_{processed_count:03d}_class_{CLASS_NAMES[orig_label]}.png")
        adv_img_path = os.path.join(OUTPUT_DIR, "adversarial_images", f"sample_{processed_count:03d}_orig_{CLASS_NAMES[orig_label]}_adv_{CLASS_NAMES[adv_label]}.png")
        
        save_image_to_disk(data, orig_img_path)
        save_image_to_disk(pert_image, adv_img_path)
        
        # Store results
        results.append({
            'idx': processed_count,
            'original_image': data.cpu(),
            'perturbation': r.cpu(),
            'perturbed_image': pert_image.cpu(),
            'original_label': orig_label,
            'adversarial_label': adv_label,
            'true_label': target.item(),
            'iterations': iterations,
            'l2_norm': l2_norm,
            'linf_norm': linf_norm,
            'relative_pert': relative_pert,
            'original_confidence': original_confidence,
            'adversarial_confidence': adv_confidence,
            'success': success,
            'orig_img_path': orig_img_path,
            'adv_img_path': adv_img_path
        })
        
        # Progress feedback
        status = "✓" if success else "✗"
        print(f"  {status} Sample {processed_count+1:3d}: {CLASS_NAMES[orig_label]:10s} → {CLASS_NAMES[adv_label]:10s} | "
              f"Iter: {iterations:2d} | L2: {l2_norm:7.4f} | L∞: {linf_norm:6.4f} | "
              f"Conf: {original_confidence:.3f}→{adv_confidence:.3f}")
        
        processed_count += 1
    
    print(f"\n{'='*60}")
    print(f"ATTACK SUMMARY")
    print(f"{'='*60}")
    print(f"Success Rate: {success_count}/{num_examples} ({100*success_count/num_examples:.1f}%)")
    print(f"Average L2 norm: {np.mean([r['l2_norm'] for r in results]):.4f}")
    print(f"Average L∞ norm: {np.mean([r['linf_norm'] for r in results]):.4f}")
    print(f"Average iterations: {np.mean([r['iterations'] for r in results]):.1f}")
    print(f"{'='*60}\n")
    
    return results

In [7]:
# ============================================
# VISUALIZATION: SINGLE ATTACK DETAILED
# ============================================
def visualize_single_attack(result, save_dir):
    """Create detailed 4-panel visualization for a single attack"""
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.patch.set_facecolor(NODE_BLACK)
    
    for ax in axes:
        ax.set_facecolor(NODE_BLACK)
        for spine in ax.spines.values():
            spine.set_edgecolor(HACKER_GREY)
    
    # Panel 1: Original image
    orig_img = denormalize(result['original_image'].squeeze()).permute(1, 2, 0).numpy()
    orig_img = np.clip(orig_img, 0, 1)
    axes[0].imshow(orig_img)
    axes[0].set_title(f"Original\nClass: {CLASS_NAMES[result['original_label']]}\nConf: {result['original_confidence']:.3f}",
                      color=HTB_GREEN, fontweight='bold', fontsize=12)
    axes[0].axis('off')
    
    # Panel 2: Perturbation (amplified)
    pert = result['perturbation'].squeeze().numpy()
    # Average across channels for visualization
    pert_vis = np.mean(pert, axis=0)
    pert_display = pert_vis - pert_vis.min()
    if pert_display.max() > 0:
        pert_display = pert_display / pert_display.max()
    
    axes[1].imshow(pert_display, cmap='inferno')
    axes[1].set_title(f"Perturbation (amplified)\nL2: {result['l2_norm']:.4f}",
                      color=NUGGET_YELLOW, fontweight='bold', fontsize=12)
    axes[1].axis('off')
    
    # Panel 3: Magnitude heatmap
    pert_mag = np.sqrt(np.sum(pert**2, axis=0))
    im = axes[2].imshow(pert_mag, cmap='viridis')
    axes[2].set_title(f"Magnitude\nL∞: {result['linf_norm']:.4f}",
                      color=AZURE, fontweight='bold', fontsize=12)
    axes[2].axis('off')
    cbar = plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
    cbar.ax.tick_params(colors=WHITE)
    
    # Panel 4: Adversarial image
    adv_img = denormalize(result['perturbed_image'].squeeze()).permute(1, 2, 0).numpy()
    adv_img = np.clip(adv_img, 0, 1)
    title_color = MALWARE_RED if result['success'] else HACKER_GREY
    axes[3].imshow(adv_img)
    axes[3].set_title(f"Adversarial\nClass: {CLASS_NAMES[result['adversarial_label']]}\nConf: {result['adversarial_confidence']:.3f}",
                      color=title_color, fontweight='bold', fontsize=12)
    axes[3].axis('off')
    
    # Summary metrics
    metrics_text = (
        f"Attack: {CLASS_NAMES[result['original_label']]} → {CLASS_NAMES[result['adversarial_label']]}  |  "
        f"Iterations: {result['iterations']}  |  "
        f"Relative Perturbation: {result['relative_pert']:.2%}"
    )
    fig.text(0.5, 0.02, metrics_text, ha='center', fontsize=11, color=WHITE, weight='bold')
    
    plt.suptitle("DeepFool Attack - Detailed Analysis", fontsize=16,
                 color=HTB_GREEN, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'detailed_attack_sample_{result["idx"]:03d}.png'),
                facecolor=NODE_BLACK, dpi=300, bbox_inches='tight')
    plt.close()



In [8]:
# ============================================
# VISUALIZATION: ATTACK GRID
# ============================================
def visualize_attack_grid(results, save_dir, n_samples=10):
    """Create grid showing original vs adversarial images"""
    print("\n[i] Generating attack grid visualization...")
    
    n_samples = min(n_samples, len(results))
    rows = n_samples
    
    fig, axes = plt.subplots(rows, 3, figsize=(15, 5 * rows))
    fig.patch.set_facecolor(NODE_BLACK)
    
    if rows == 1:
        axes = axes.reshape(1, -1)
    
    for ax_row in axes:
        for ax in ax_row:
            ax.set_facecolor(NODE_BLACK)
            for spine in ax.spines.values():
                spine.set_edgecolor(HACKER_GREY)
    
    for idx in range(n_samples):
        result = results[idx]
        
        # Column 0: Original
        orig_img = denormalize(result['original_image'].squeeze()).permute(1, 2, 0).numpy()
        orig_img = np.clip(orig_img, 0, 1)
        axes[idx, 0].imshow(orig_img)
        axes[idx, 0].set_title(f"Original: {CLASS_NAMES[result['original_label']]}\nConf: {result['original_confidence']:.3f}",
                               color=HTB_GREEN, fontsize=10, fontweight='bold')
        axes[idx, 0].axis('off')
        
        # Column 1: Perturbation heatmap
        pert = result['perturbation'].squeeze().numpy()
        pert_mag = np.sqrt(np.sum(pert**2, axis=0))
        im = axes[idx, 1].imshow(pert_mag, cmap='hot')
        axes[idx, 1].set_title(f"L2: {result['l2_norm']:.4f}\nIter: {result['iterations']}",
                               color=NUGGET_YELLOW, fontsize=10, fontweight='bold')
        axes[idx, 1].axis('off')
        
        # Column 2: Adversarial
        adv_img = denormalize(result['perturbed_image'].squeeze()).permute(1, 2, 0).numpy()
        adv_img = np.clip(adv_img, 0, 1)
        title_color = MALWARE_RED if result['success'] else HACKER_GREY
        axes[idx, 2].imshow(adv_img)
        axes[idx, 2].set_title(f"Adversarial: {CLASS_NAMES[result['adversarial_label']]}\nConf: {result['adversarial_confidence']:.3f}",
                               color=title_color, fontsize=10, fontweight='bold')
        axes[idx, 2].axis('off')
    
    plt.suptitle('DeepFool Attack Grid: Original → Perturbation → Adversarial',
                 color=HTB_GREEN, fontsize=18, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'attack_grid.png'),
                facecolor=NODE_BLACK, dpi=200, bbox_inches='tight')
    plt.close()
    print("[✓] Attack grid saved")



In [9]:
# ============================================
# VISUALIZATION: PERTURBATION ANALYSIS
# ============================================
def visualize_perturbation_analysis(results, save_dir, n_samples=6):
    """Analyze perturbation patterns across multiple samples"""
    print("\n[i] Generating perturbation analysis...")
    
    successful = [r for r in results if r['success']][:n_samples]
    n_samples = len(successful)
    
    fig, axes = plt.subplots(2, n_samples, figsize=(5*n_samples, 10))
    fig.patch.set_facecolor(NODE_BLACK)
    
    if n_samples == 1:
        axes = axes.reshape(2, 1)
    
    for ax_row in axes:
        for ax in ax_row:
            ax.set_facecolor(NODE_BLACK)
            for spine in ax.spines.values():
                spine.set_edgecolor(HACKER_GREY)
    
    for idx, result in enumerate(successful):
        # Top row: Raw perturbation heatmap
        pert = result['perturbation'].squeeze().numpy()
        pert_avg = np.mean(pert, axis=0)
        vmax = np.abs(pert_avg).max() or 1e-6
        
        im_top = axes[0, idx].imshow(pert_avg, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        axes[0, idx].set_title(f'Perturbation\nL2={result["l2_norm"]:.3f}',
                               color=HTB_GREEN, fontsize=10, fontweight='bold')
        axes[0, idx].axis('off')
        
        cbar = plt.colorbar(im_top, ax=axes[0, idx], fraction=0.046, pad=0.04)
        cbar.ax.tick_params(colors=WHITE, labelsize=8)
        
        # Bottom row: Amplified difference
        orig_img = result['original_image'].squeeze().numpy()
        adv_img = result['perturbed_image'].squeeze().detach().numpy()
        diff_amplified = (adv_img - orig_img) * 10
        diff_avg = np.mean(diff_amplified, axis=0)
        
        im_bottom = axes[1, idx].imshow(diff_avg, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
        axes[1, idx].set_title(f"{CLASS_NAMES[result['original_label']]} → {CLASS_NAMES[result['adversarial_label']]}\n({result['iterations']} iters)",
                               color=NUGGET_YELLOW, fontsize=10, fontweight='bold')
        axes[1, idx].axis('off')
        
        cbar = plt.colorbar(im_bottom, ax=axes[1, idx], fraction=0.046, pad=0.04)
        cbar.ax.tick_params(colors=WHITE, labelsize=8)
    
    plt.suptitle('DeepFool Perturbation Analysis', color=HTB_GREEN, fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'perturbation_analysis.png'),
                facecolor=NODE_BLACK, dpi=200, bbox_inches='tight')
    plt.close()
    print("[✓] Perturbation analysis saved")


In [10]:
# ============================================
# VISUALIZATION: METRICS
# ============================================
def visualize_metrics(results, save_dir):
    """Visualize attack metrics statistics"""
    print("\n[i] Generating metrics visualization...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.patch.set_facecolor(NODE_BLACK)
    
    for ax_row in axes:
        for ax in ax_row:
            ax.set_facecolor(NODE_BLACK)
            for spine in ax.spines.values():
                spine.set_edgecolor(HACKER_GREY)
            ax.tick_params(colors=WHITE)
            ax.grid(True, alpha=0.3, color=HACKER_GREY, linestyle='--')
    
    # Panel 1: L2 Norm Distribution
    l2_norms = [r['l2_norm'] for r in results]
    axes[0, 0].hist(l2_norms, bins=20, color=HTB_GREEN, alpha=0.7, edgecolor=HACKER_GREY, linewidth=1.5)
    axes[0, 0].set_xlabel('L2 Norm', color=WHITE, fontsize=12, fontweight='bold')
    axes[0, 0].set_ylabel('Frequency', color=WHITE, fontsize=12, fontweight='bold')
    axes[0, 0].set_title('Perturbation L2 Magnitude Distribution', color=HTB_GREEN, fontsize=14, fontweight='bold')
    axes[0, 0].axvline(np.mean(l2_norms), color=MALWARE_RED, linestyle='--', linewidth=2, label=f'Mean: {np.mean(l2_norms):.3f}')
    axes[0, 0].legend(facecolor=NODE_BLACK, edgecolor=HACKER_GREY, labelcolor=WHITE)
    
    # Panel 2: Iteration Count Distribution
    iterations = [r['iterations'] for r in results]
    axes[0, 1].hist(iterations, bins=range(1, max(iterations)+2), color=AZURE, alpha=0.7, edgecolor=HACKER_GREY, linewidth=1.5)
    axes[0, 1].set_xlabel('Iterations', color=WHITE, fontsize=12, fontweight='bold')
    axes[0, 1].set_ylabel('Frequency', color=WHITE, fontsize=12, fontweight='bold')
    axes[0, 1].set_title('Iterations Required for Success', color=HTB_GREEN, fontsize=14, fontweight='bold')
    axes[0, 1].axvline(np.mean(iterations), color=MALWARE_RED, linestyle='--', linewidth=2, label=f'Mean: {np.mean(iterations):.1f}')
    axes[0, 1].legend(facecolor=NODE_BLACK, edgecolor=HACKER_GREY, labelcolor=WHITE)
    
    # Panel 3: Per-Class Success Rate
    class_success = {}
    for r in results:
        orig = r['original_label']
        if orig not in class_success:
            class_success[orig] = {'total': 0, 'success': 0}
        class_success[orig]['total'] += 1
        if r['success']:
            class_success[orig]['success'] += 1
    
    classes = sorted(class_success.keys())
    success_rates = [
        class_success[c]['success'] / class_success[c]['total'] * 100
        if class_success[c]['total'] > 0 else 0
        for c in classes
    ]
    
    bars = axes[1, 0].bar([CLASS_NAMES[c] for c in classes], success_rates,
                          color=NUGGET_YELLOW, alpha=0.7, edgecolor=HACKER_GREY, linewidth=1.5)
    axes[1, 0].set_xlabel('Original Class', color=WHITE, fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Success Rate (%)', color=WHITE, fontsize=12, fontweight='bold')
    axes[1, 0].set_title('Attack Success Rate by Class', color=HTB_GREEN, fontsize=14, fontweight='bold')
    axes[1, 0].set_ylim(0, 105)
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    for bar, rate in zip(bars, success_rates):
        height = bar.get_height()
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 1,
                        f'{rate:.0f}%', ha='center', va='bottom',
                        color=WHITE, fontsize=9, fontweight='bold')
    
    # Panel 4: Confidence Change
    conf_changes = [(r['original_confidence'] - r['adversarial_confidence']) * 100 for r in results if r['success']]
    axes[1, 1].hist(conf_changes, bins=20, color=PINK, alpha=0.7, edgecolor=HACKER_GREY, linewidth=1.5)
    axes[1, 1].set_xlabel('Confidence Drop (%)', color=WHITE, fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel('Frequency', color=WHITE, fontsize=12, fontweight='bold')
    axes[1, 1].set_title('Model Confidence Change', color=HTB_GREEN, fontsize=14, fontweight='bold')
    axes[1, 1].axvline(np.mean(conf_changes), color=MALWARE_RED, linestyle='--', linewidth=2, label=f'Mean: {np.mean(conf_changes):.1f}%')
    axes[1, 1].legend(facecolor=NODE_BLACK, edgecolor=HACKER_GREY, labelcolor=WHITE)
    
    plt.suptitle('DeepFool Attack Metrics', color=HTB_GREEN, fontsize=18, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'attack_metrics.png'),
                facecolor=NODE_BLACK, dpi=200, bbox_inches='tight')
    plt.close()
    print("[✓] Metrics visualization saved")


In [11]:


# ============================================
# PRINT SUMMARY STATISTICS
# ============================================
def print_summary_statistics(results):
    """Print detailed summary statistics"""
    print(f"\n{'='*60}")
    print("DETAILED ATTACK STATISTICS")
    print(f"{'='*60}")
    
    successful = [r for r in results if r['success']]
    
    if successful:
        print(f"\nOverall Performance:")
        print(f"  Success Rate: {len(successful)}/{len(results)} ({100*len(successful)/len(results):.1f}%)")
        print(f"  Average L2 Norm: {np.mean([r['l2_norm'] for r in successful]):.4f}")
        print(f"  L2 Range: [{min([r['l2_norm'] for r in successful]):.4f}, {max([r['l2_norm'] for r in successful]):.4f}]")
        print(f"  Average L∞ Norm: {np.mean([r['linf_norm'] for r in successful]):.4f}")
        print(f"  Average Iterations: {np.mean([r['iterations'] for r in successful]):.1f}")
        print(f"  Average Relative Perturbation: {np.mean([r['relative_pert'] for r in successful]):.2%}")
        
        # Confidence analysis
        conf_drops = [r['original_confidence'] - r['adversarial_confidence'] for r in successful]
        print(f"\nConfidence Analysis:")
        print(f"  Average Confidence Drop: {np.mean(conf_drops):.3f}")
        print(f"  Original Confidence (avg): {np.mean([r['original_confidence'] for r in successful]):.3f}")
        print(f"  Adversarial Confidence (avg): {np.mean([r['adversarial_confidence'] for r in successful]):.3f}")
        
        # Class transitions
        transitions = {}
        for r in successful:
            key = f"{CLASS_NAMES[r['original_label']]} → {CLASS_NAMES[r['adversarial_label']]}"
            transitions[key] = transitions.get(key, 0) + 1
        
        print(f"\nMost Common Misclassifications:")
        for trans, count in sorted(transitions.items(), key=lambda x: x[1], reverse=True)[:10]:
            print(f"  {trans:30s}: {count} times")
        
        # Per-class statistics
        print(f"\nPer-Class Statistics:")
        for class_idx in range(8):
            class_results = [r for r in successful if r['original_label'] == class_idx]
            if class_results:
                avg_l2 = np.mean([r['l2_norm'] for r in class_results])
                avg_iter = np.mean([r['iterations'] for r in class_results])
                print(f"  {CLASS_NAMES[class_idx]:10s}: L2={avg_l2:7.4f}, Iter={avg_iter:4.1f}, Count={len(class_results):2d}")
    
    print(f"{'='*60}\n")



In [12]:
# ============================================
# CLASS SELECTION
# ============================================
def select_class():
    """Interactive class selection for targeted attacks"""
    print("\n" + "="*60)
    print("MALWARE CLASS SELECTION")
    print("="*60)
    print("\nAvailable malware classes:")
    for idx, class_name in enumerate(CLASS_NAMES):
        print(f"  [{idx}] {class_name}")
    print(f"  [8] ALL CLASSES (no filter)")
    print("="*60)
    
    while True:
        try:
            choice = input("\nEnter class number [0-8]: ").strip()
            choice = int(choice)
            
            if 0 <= choice <= 8:
                if choice == 8:
                    print(f"\n[✓] Selected: ALL CLASSES (no filtering)")
                    return None
                else:
                    print(f"\n[✓] Selected: {CLASS_NAMES[choice]}")
                    return choice
            else:
                print("[!] Invalid choice. Please enter a number between 0 and 8.")
        except ValueError:
            print("[!] Invalid input. Please enter a number.")
        except KeyboardInterrupt:
            print("\n\n[!] Selection cancelled.")
            exit(0)


In [13]:
# ============================================
# MAIN EXECUTION
# ============================================
def main():
    # Device setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[i] Using device: {device}")
    
    # Load model and data
    model, test_loader = load_model_and_data(MODEL_PATH, DATA_PATH, device)
    
    # Select target class
    target_class = select_class()
    
    # Generate batch attacks
    results = generate_batch_attacks(model, test_loader, device, NUM_EXAMPLES, target_class)
    
    # Print summary statistics
    print_summary_statistics(results)
    
    # Generate visualizations
    print(f"\n{'='*60}")
    print("GENERATING VISUALIZATIONS")
    print(f"{'='*60}")
    
    # 1. Detailed view of first successful attack
    successful_results = [r for r in results if r['success']]
    if successful_results:
        print("\n[i] Creating detailed analysis for first successful attack...")
        visualize_single_attack(successful_results[0], OUTPUT_DIR)
        print("[✓] Detailed attack visualization saved")
    
    # 2. Attack grid
    visualize_attack_grid(results, OUTPUT_DIR, n_samples=10)
    
    # 3. Perturbation analysis
    visualize_perturbation_analysis(results, OUTPUT_DIR, n_samples=6)
    
    # 4. Metrics visualization
    visualize_metrics(results, OUTPUT_DIR)
    
    print(f"\n{'='*60}")
    print("✓ ALL VISUALIZATIONS COMPLETE")
    print(f"{'='*60}")
    print(f"\nGenerated files in '{OUTPUT_DIR}/':")
    print("  Visualizations:")
    print("    • attack_grid.png")
    print("    • perturbation_analysis.png")
    print("    • attack_metrics.png")
    print("    • detailed_attack_sample_000.png")
    print(f"\n  Adversarial Images: {len(results)} images in 'adversarial_images/'")
    print(f"  Original Images: {len(results)} images in 'original_images/'")
    print(f"\n{'='*60}\n")

if __name__ == "__main__":
    main()

[i] Using device: cpu
[i] Loading model...
[✓] Model loaded successfully
[i] Loading test dataset...
[✓] Test dataset loaded: 2209 samples

MALWARE CLASS SELECTION

Available malware classes:
  [0] adware
  [1] backdoor
  [2] benign
  [3] downloader
  [4] spyware
  [5] trojan
  [6] virus
  [7] worm
  [8] ALL CLASSES (no filter)



Enter class number [0-8]:  5



[✓] Selected: trojan

GENERATING 30 ADVERSARIAL EXAMPLES
TARGET CLASS: TROJAN

  ✓ Sample   1: trojan     → benign     | Iter:  5 | L2:  0.0258 | L∞: 0.0013 | Conf: 0.952→0.455
  ✓ Sample   2: benign     → worm       | Iter:  5 | L2:  0.0168 | L∞: 0.0009 | Conf: 0.956→0.473
  ✓ Sample   3: trojan     → backdoor   | Iter:  2 | L2:  0.0008 | L∞: 0.0001 | Conf: 0.516→0.502
  ✓ Sample   4: trojan     → benign     | Iter:  4 | L2:  0.0169 | L∞: 0.0017 | Conf: 0.972→0.492
  ✓ Sample   5: trojan     → adware     | Iter:  5 | L2:  0.0352 | L∞: 0.0024 | Conf: 0.999→0.504
  ✓ Sample   6: trojan     → benign     | Iter:  5 | L2:  0.0347 | L∞: 0.0018 | Conf: 0.999→0.492
  ✓ Sample   7: trojan     → benign     | Iter:  5 | L2:  0.0107 | L∞: 0.0007 | Conf: 0.783→0.455
  ✓ Sample   8: trojan     → backdoor   | Iter:  5 | L2:  0.0269 | L∞: 0.0016 | Conf: 0.853→0.491
  ✓ Sample   9: trojan     → backdoor   | Iter:  5 | L2:  0.0269 | L∞: 0.0016 | Conf: 0.853→0.491
  ✓ Sample  10: trojan     → benign   